# Práctica Semana 1 - Proyecto ETL con datos WiFi CSI

## Tema del proyecto

El presente proyecto tiene como objetivo realizar un análisis inicial de señales WiFi CSI orientadas a la detección de presencia, movimiento y actividad humana. Este enfoque puede apoyar controles de seguridad física en áreas restringidas, mediante el uso de señales inalámbricas como fuente de datos para identificar patrones de comportamiento.

## Fuentes de datos

Para la práctica se utilizarán tres fuentes principales:

1. Dataset WIFI CSI object detection, cargado posteriormente en PostgreSQL.
2. Dataset WiFi CSI Human Activity Recognition, procesado desde un archivo JSON.
3. Archivo annotations.csv del dataset HAR-ORT, asociado a capturas PCAP de actividades humanas.

# Fase 1: Selección y organización de fuentes de datos

En esta fase se seleccionaron las fuentes de datos que serán utilizadas para el proyecto ETL. 
El dominio elegido fue el análisis de señales WiFi CSI para la detección de presencia, 
movimiento y actividad humana como apoyo a controles de seguridad física.

Se utilizarán tres fuentes principales:

1. Dataset WIFI CSI object detection, almacenado en `data/postgres/raw_data.csv`.
2. Dataset WiFi CSI Human Activity Recognition, almacenado en `data/json/data_apml_min.json`.
3. Dataset HAR-ORT, mediante el archivo `data/har_ort/annotations.csv`, asociado a capturas PCAP.

Los archivos PCAP se consideran datos crudos de captura inalámbrica. Para esta fase de la práctica, 
se utilizará el archivo `annotations.csv` como fuente tabular estructurada.

# Fase 2: Configuración del entorno con Docker y PostgreSQL

En esta fase se configuró un contenedor Docker con PostgreSQL utilizando un archivo `docker-compose.yml`. 
Las credenciales de conexión fueron separadas en el archivo `.env`, siguiendo una práctica adecuada 
para no escribir credenciales directamente dentro del archivo de composición.

El contenedor creado permite alojar la base de datos `wifi_csi_db`, donde posteriormente será cargado 
el dataset principal `raw_data.csv`.

Como evidencia de esta fase se generaron capturas de la ejecución del comando `docker compose up -d`, 
del contenedor activo mediante `docker ps` y de la visualización del contenedor en Docker Desktop.

# Fase 3: Conexión a PostgreSQL y carga del dataset principal

En esta fase se realiza la conexión a la base de datos PostgreSQL ejecutada mediante Docker. 
Posteriormente, se carga el archivo `raw_data.csv`, correspondiente al dataset WIFI CSI object detection, 
en una tabla denominada `wifi_csi_object_detection`.

Esta fuente será utilizada como el primer DataFrame principal proveniente de PostgreSQL.

In [2]:
from pathlib import Path

print("Ruta actual del notebook:")
print(Path.cwd())

print("\nArchivos en la raíz del proyecto:")
for archivo in Path(".").iterdir():
    print(archivo)

Ruta actual del notebook:
c:\Users\Manuel\Desktop\proyectoWifi

Archivos en la raíz del proyecto:
.env
data
desarrolloGrupo10.ipynb
docker-compose.yml
evidencias
scripts
venv


In [8]:
import os
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from dotenv import load_dotenv

# Cargar variables del archivo .env
load_dotenv(dotenv_path=Path(".env"), override=True, encoding="utf-8")

POSTGRES_USER = os.getenv("POSTGRES_USER")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD")
POSTGRES_DB = os.getenv("POSTGRES_DB")
POSTGRES_PORT = os.getenv("POSTGRES_PORT")

print("Usuario:", POSTGRES_USER)
print("Base de datos:", POSTGRES_DB)
print("Puerto:", POSTGRES_PORT)
print("Contraseña cargada:", repr(POSTGRES_PASSWORD))

connection_url = URL.create(
    drivername="postgresql+pg8000",
    username=POSTGRES_USER,
    password=POSTGRES_PASSWORD,
    host="127.0.0.1",
    port=int(POSTGRES_PORT),
    database=POSTGRES_DB
)

engine = create_engine(connection_url)

with engine.connect() as conn:
    result = conn.execute(text("SELECT current_database();"))
    print("Conectado a la base de datos:", result.scalar())

Usuario: postgres
Base de datos: wifi_csi_db
Puerto: 5433
Contraseña cargada: 'postgres123'
Conectado a la base de datos: wifi_csi_db


## Lectura del archivo CSV principal

Se carga el archivo `raw_data.csv`, correspondiente al dataset WIFI CSI object detection. 
Este archivo contiene mediciones de señales WiFi CSI, incluyendo variables numéricas de amplitud y fase, 
junto con variables categóricas relacionadas con el tipo, objeto, posición y configuración.

In [9]:
from pathlib import Path
import pandas as pd

ruta_raw_data = Path("data/postgres/raw_data.csv")

print("¿Existe el archivo?", ruta_raw_data.exists())
print("Ruta:", ruta_raw_data)

df_raw = pd.read_csv(ruta_raw_data)

print("Filas y columnas:", df_raw.shape)
df_raw.head()

¿Existe el archivo? True
Ruta: data\postgres\raw_data.csv
Filas y columnas: (56939, 109)


,amp_0,amp_1,amp_2,amp_3,amp_4,amp_5,amp_6,amp_7,amp_8,amp_9,...,phase_47,phase_48,phase_49,phase_50,phase_51,type,day,object_id,position,configuration
0,14.317821,15.524175,14.317821,15.524175,15.524175,14.866069,16.155494,14.866069,16.552945,15.652476,...,1.712693,1.723446,1.781890,1.781890,1.735945,organic,1,0,A10,COMPRIDO
1,19.209373,19.798990,18.439089,21.931712,20.518285,19.798990,20.518285,21.400935,22.203603,22.203603,...,2.010639,1.951303,1.951303,2.007423,2.034444,organic,1,0,A10,COMPRIDO
2,15.132746,15.132746,14.142136,15.132746,16.124515,15.033296,16.000000,16.031220,16.000000,16.031220,...,1.107149,1.176005,1.107149,1.071450,1.144169,organic,1,0,A10,COMPRIDO
3,12.649111,13.928388,13.000000,13.928388,14.317821,14.317821,15.652476,15.264338,15.264338,14.764823,...,0.083141,0.165149,0.076772,0.165149,0.165149,organic,1,0,A10,COMPRIDO
4,18.439089,19.416488,19.416488,21.587033,20.024984,21.213203,21.095023,22.022716,22.022716,22.022716,...,2.601173,2.651635,2.570255,2.601173,2.622447,organic,1,0,A10,COMPRIDO


In [10]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 56939 entries, 0 to 56938
Columns: 109 entries, amp_0 to configuration
dtypes: float64(104), int64(2), str(3)
memory usage: 47.4 MB


In [11]:
df_raw.columns.tolist()

['amp_0',
 'amp_1',
 'amp_2',
 'amp_3',
 'amp_4',
 'amp_5',
 'amp_6',
 'amp_7',
 'amp_8',
 'amp_9',
 'amp_10',
 'amp_11',
 'amp_12',
 'amp_13',
 'amp_14',
 'amp_15',
 'amp_16',
 'amp_17',
 'amp_18',
 'amp_19',
 'amp_20',
 'amp_21',
 'amp_22',
 'amp_23',
 'amp_24',
 'amp_25',
 'amp_26',
 'amp_27',
 'amp_28',
 'amp_29',
 'amp_30',
 'amp_31',
 'amp_32',
 'amp_33',
 'amp_34',
 'amp_35',
 'amp_36',
 'amp_37',
 'amp_38',
 'amp_39',
 'amp_40',
 'amp_41',
 'amp_42',
 'amp_43',
 'amp_44',
 'amp_45',
 'amp_46',
 'amp_47',
 'amp_48',
 'amp_49',
 'amp_50',
 'amp_51',
 'phase_0',
 'phase_1',
 'phase_2',
 'phase_3',
 'phase_4',
 'phase_5',
 'phase_6',
 'phase_7',
 'phase_8',
 'phase_9',
 'phase_10',
 'phase_11',
 'phase_12',
 'phase_13',
 'phase_14',
 'phase_15',
 'phase_16',
 'phase_17',
 'phase_18',
 'phase_19',
 'phase_20',
 'phase_21',
 'phase_22',
 'phase_23',
 'phase_24',
 'phase_25',
 'phase_26',
 'phase_27',
 'phase_28',
 'phase_29',
 'phase_30',
 'phase_31',
 'phase_32',
 'phase_33',
 'phas

## Carga del dataset principal en PostgreSQL

Luego de validar la estructura del archivo CSV, se carga el DataFrame en PostgreSQL como una tabla denominada 
`wifi_csi_object_detection`. Esta tabla representa la fuente obligatoria de base de datos de la práctica.

In [12]:
df_raw.to_sql(
    "wifi_csi_object_detection",
    engine,
    if_exists="replace",
    index=False
)

print("Tabla wifi_csi_object_detection creada correctamente en PostgreSQL.")

Tabla wifi_csi_object_detection creada correctamente en PostgreSQL.


In [13]:
from sqlalchemy import text

with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'public';
    """))

    tablas = result.fetchall()

tablas

[('wifi_csi_object_detection',)]

## DataFrame 1: Fuente PostgreSQL

Se consulta la tabla `wifi_csi_object_detection` desde PostgreSQL para generar el primer DataFrame principal de la práctica.

In [14]:
df_postgres = pd.read_sql(
    "SELECT * FROM wifi_csi_object_detection",
    engine
)

print("Filas y columnas:", df_postgres.shape)
df_postgres.head()

Filas y columnas: (56939, 109)


,amp_0,amp_1,amp_2,amp_3,amp_4,amp_5,amp_6,amp_7,amp_8,amp_9,...,phase_47,phase_48,phase_49,phase_50,phase_51,type,day,object_id,position,configuration
0,14.317821,15.524175,14.317821,15.524175,15.524175,14.866069,16.155494,14.866069,16.552945,15.652476,...,1.712693,1.723446,1.781890,1.781890,1.735945,organic,1,0,A10,COMPRIDO
1,19.209373,19.798990,18.439089,21.931712,20.518285,19.798990,20.518285,21.400935,22.203603,22.203603,...,2.010639,1.951303,1.951303,2.007423,2.034444,organic,1,0,A10,COMPRIDO
2,15.132746,15.132746,14.142136,15.132746,16.124515,15.033296,16.000000,16.031220,16.000000,16.031220,...,1.107149,1.176005,1.107149,1.071450,1.144169,organic,1,0,A10,COMPRIDO
3,12.649111,13.928388,13.000000,13.928388,14.317821,14.317821,15.652476,15.264338,15.264338,14.764823,...,0.083141,0.165149,0.076772,0.165149,0.165149,organic,1,0,A10,COMPRIDO
4,18.439089,19.416488,19.416488,21.587033,20.024984,21.213203,21.095023,22.022716,22.022716,22.022716,...,2.601173,2.651635,2.570255,2.601173,2.622447,organic,1,0,A10,COMPRIDO


In [15]:
df_raw.to_sql(
    "wifi_csi_object_detection",
    engine,
    if_exists="replace",
    index=False
)

print("Tabla wifi_csi_object_detection creada correctamente en PostgreSQL.")

Tabla wifi_csi_object_detection creada correctamente en PostgreSQL.


In [16]:
from sqlalchemy import text

with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'public';
    """))

    tablas = result.fetchall()

tablas

[('wifi_csi_object_detection',)]

In [17]:
df_postgres = pd.read_sql(
    "SELECT * FROM wifi_csi_object_detection",
    engine
)

print("Filas y columnas:", df_postgres.shape)
df_postgres.head()

Filas y columnas: (56939, 109)


,amp_0,amp_1,amp_2,amp_3,amp_4,amp_5,amp_6,amp_7,amp_8,amp_9,...,phase_47,phase_48,phase_49,phase_50,phase_51,type,day,object_id,position,configuration
0,14.317821,15.524175,14.317821,15.524175,15.524175,14.866069,16.155494,14.866069,16.552945,15.652476,...,1.712693,1.723446,1.781890,1.781890,1.735945,organic,1,0,A10,COMPRIDO
1,19.209373,19.798990,18.439089,21.931712,20.518285,19.798990,20.518285,21.400935,22.203603,22.203603,...,2.010639,1.951303,1.951303,2.007423,2.034444,organic,1,0,A10,COMPRIDO
2,15.132746,15.132746,14.142136,15.132746,16.124515,15.033296,16.000000,16.031220,16.000000,16.031220,...,1.107149,1.176005,1.107149,1.071450,1.144169,organic,1,0,A10,COMPRIDO
3,12.649111,13.928388,13.000000,13.928388,14.317821,14.317821,15.652476,15.264338,15.264338,14.764823,...,0.083141,0.165149,0.076772,0.165149,0.165149,organic,1,0,A10,COMPRIDO
4,18.439089,19.416488,19.416488,21.587033,20.024984,21.213203,21.095023,22.022716,22.022716,22.022716,...,2.601173,2.651635,2.570255,2.601173,2.622447,organic,1,0,A10,COMPRIDO


# Fase 4: Generación de los tres DataFrames principales

En esta fase se generan los tres DataFrames principales de la práctica. 
El primer DataFrame proviene de PostgreSQL, el segundo se construye a partir de un archivo JSON 
con señales WiFi CSI y el tercero se obtiene desde un archivo CSV de anotaciones asociado a capturas PCAP.

## DataFrame 2: Fuente JSON - WiFi CSI Human Activity Recognition

El archivo `data_apml_min.json` contiene mediciones de señales WiFi CSI en formato matricial. 
Para facilitar su análisis con pandas, cada muestra será resumida mediante estadísticas descriptivas 
como promedio, máximo, mínimo y desviación estándar.

In [ ]:
import json
import numpy as np

ruta_json = Path("data/json/data_apml_min.json")

print("¿Existe el archivo JSON?", ruta_json.exists())
print("Ruta:", ruta_json)

with open(ruta_json, "r", encoding="utf-8") as file:
    data_json = json.load(file)

print("Cantidad de muestras:", len(data_json))
print("Dimensión de la primera muestra:", np.array(data_json[0]).shape)

¿Existe el archivo JSON? True
Ruta: data\json\data_apml_min.json
Cantidad de muestras: 21298
Dimensión de la primera muestra: (9, 56)


In [ ]:
registros = []

for i, muestra in enumerate(data_json):
    matriz = np.array(muestra)

    registros.append({
        "sample_id": i + 1,
        "csi_mean": matriz.mean(),
        "csi_max": matriz.max(),
        "csi_min": matriz.min(),
        "csi_std": matriz.std(),
        "num_filas": matriz.shape[0],
        "num_columnas": matriz.shape[1]
    })

df_har = pd.DataFrame(registros)

print("Filas y columnas:", df_har.shape)
df_har.head()

Filas y columnas: (21298, 7)


,sample_id,csi_mean,csi_max,csi_min,csi_std,num_filas,num_columnas
0,1,92.915050,176.139150,17.464249,41.452590,9,56
1,2,87.472571,165.423698,26.683328,33.616029,9,56
2,3,94.036743,172.699160,23.086793,38.146070,9,56
3,4,99.896125,166.207701,36.055513,31.917030,9,56
4,5,110.250978,184.390889,40.049969,34.661883,9,56


## DataFrame 3: Fuente CSV - Anotaciones HAR-ORT

El archivo `annotations.csv` contiene las etiquetas de actividad humana asociadas a los archivos PCAP. 
Estas etiquetas permiten identificar la actividad realizada, el usuario participante y el archivo de captura correspondiente.

In [ ]:
ruta_annotations = Path("data/har_ort/annotations.csv")

print("¿Existe el archivo CSV de anotaciones?", ruta_annotations.exists())
print("Ruta:", ruta_annotations)

df_annotations = pd.read_csv(ruta_annotations)

print("Filas y columnas:", df_annotations.shape)
df_annotations.head()

¿Existe el archivo CSV de anotaciones? True
Ruta: data\har_ort\annotations.csv
Filas y columnas: (1527, 4)


,id,activity,user,filename
0,1,fall,benja,fall_benja_2025-07-12_00-23-38.pcap
1,2,fall,benja,fall_benja_2025-07-12_00-23-47.pcap
2,3,fall,benja,fall_benja_2025-07-12_00-23-56.pcap
3,4,fall,benja,fall_benja_2025-07-12_00-24-05.pcap
4,5,fall,benja,fall_benja_2025-07-12_00-24-14.pcap


In [ ]:
df_annotations["filename_length"] = df_annotations["filename"].str.len()
df_annotations["is_pcap"] = df_annotations["filename"].str.endswith(".pcap")

df_annotations.head()

,id,activity,user,filename,filename_length,is_pcap
0,1,fall,benja,fall_benja_2025-07-12_00-23-38.pcap,35,True
1,2,fall,benja,fall_benja_2025-07-12_00-23-47.pcap,35,True
2,3,fall,benja,fall_benja_2025-07-12_00-23-56.pcap,35,True
3,4,fall,benja,fall_benja_2025-07-12_00-24-05.pcap,35,True
4,5,fall,benja,fall_benja_2025-07-12_00-24-14.pcap,35,True


# Fase 5: Análisis exploratorio de los DataFrames

En esta fase se realiza el análisis exploratorio inicial de los tres DataFrames principales. 
El objetivo es identificar la estructura general de los datos, revisar valores nulos, calcular 
estadísticas descriptivas básicas y generar agrupaciones que permitan interpretar patrones 
en las señales WiFi CSI y en las actividades humanas registradas.

## Análisis del DataFrame 1: `df_postgres`

Este DataFrame proviene de la tabla `wifi_csi_object_detection` almacenada en PostgreSQL. 
Contiene mediciones de amplitud y fase de señales WiFi CSI, además de variables categóricas 
como tipo, día, objeto, posición y configuración.

In [ ]:
df_postgres.head()

,amp_0,amp_1,amp_2,amp_3,amp_4,amp_5,amp_6,amp_7,amp_8,amp_9,...,phase_47,phase_48,phase_49,phase_50,phase_51,type,day,object_id,position,configuration
0,14.317821,15.524175,14.317821,15.524175,15.524175,14.866069,16.155494,14.866069,16.552945,15.652476,...,1.712693,1.723446,1.781890,1.781890,1.735945,organic,1,0,A10,COMPRIDO
1,19.209373,19.798990,18.439089,21.931712,20.518285,19.798990,20.518285,21.400935,22.203603,22.203603,...,2.010639,1.951303,1.951303,2.007423,2.034444,organic,1,0,A10,COMPRIDO
2,15.132746,15.132746,14.142136,15.132746,16.124515,15.033296,16.000000,16.031220,16.000000,16.031220,...,1.107149,1.176005,1.107149,1.071450,1.144169,organic,1,0,A10,COMPRIDO
3,12.649111,13.928388,13.000000,13.928388,14.317821,14.317821,15.652476,15.264338,15.264338,14.764823,...,0.083141,0.165149,0.076772,0.165149,0.165149,organic,1,0,A10,COMPRIDO
4,18.439089,19.416488,19.416488,21.587033,20.024984,21.213203,21.095023,22.022716,22.022716,22.022716,...,2.601173,2.651635,2.570255,2.601173,2.622447,organic,1,0,A10,COMPRIDO


In [ ]:
df_postgres.isnull().any()

amp_0            False
amp_1            False
amp_2            False
amp_3            False
amp_4            False
                 ...  
type             False
day              False
object_id        False
position         False
configuration    False
Length: 109, dtype: bool

In [ ]:
df_postgres.isnull().sum()

amp_0            0
amp_1            0
amp_2            0
amp_3            0
amp_4            0
                ..
type             0
day              0
object_id        0
position         0
configuration    0
Length: 109, dtype: int64

In [ ]:
estadisticas_amp_0 = {
    "promedio_amp_0": df_postgres["amp_0"].mean(),
    "maximo_amp_0": df_postgres["amp_0"].max(),
    "minimo_amp_0": df_postgres["amp_0"].min()
}

estadisticas_amp_0

{'promedio_amp_0': np.float64(17.116658329152926),
 'maximo_amp_0': np.float64(97.49871794028884),
 'minimo_amp_0': np.float64(0.0)}

In [ ]:
agrupacion_postgres = df_postgres.groupby(["type", "position"])["amp_0"].agg(["max", "min"])

agrupacion_postgres

max        min
type    position                      
empty   NONE      97.498718   6.708204
metalic A16       45.803930   9.486833
        C14       32.015621   4.000000
        C19       40.199502   6.324555
        C6        64.660653   1.000000
        D10       38.470768   1.000000
        F12       30.083218   2.236068
        F3        63.324561  11.045361
organic A10       47.000000   8.544004
        A16       30.149627   1.000000
        B21       22.022716  11.045361
        C14       36.055513   3.605551
        C19       49.244289   0.000000
        C6        31.240999   7.211103
        D10       41.194660   2.236068
        E17       44.418465  13.038405
        F12       31.764760   2.000000
        F3        26.172505  10.000000

El análisis permite observar el comportamiento de la variable `amp_0` según el tipo de objeto 
y la posición registrada. Esta agrupación ayuda a identificar variaciones en la señal WiFi CSI 
dependiendo de las condiciones de captura.

## Análisis del DataFrame 2: `df_har`

Este DataFrame fue construido a partir del archivo JSON del dataset WiFi CSI Human Activity Recognition. 
Cada muestra fue resumida mediante estadísticas como promedio, máximo, mínimo y desviación estándar 
de las señales CSI.

In [ ]:
df_har["rango_csi"] = pd.cut(
    df_har["csi_mean"],
    bins=3,
    labels=["bajo", "medio", "alto"]
)

df_har.head()

,sample_id,csi_mean,csi_max,csi_min,csi_std,num_filas,num_columnas,rango_csi
0,1,92.915050,176.139150,17.464249,41.452590,9,56,medio
1,2,87.472571,165.423698,26.683328,33.616029,9,56,medio
2,3,94.036743,172.699160,23.086793,38.146070,9,56,medio
3,4,99.896125,166.207701,36.055513,31.917030,9,56,medio
4,5,110.250978,184.390889,40.049969,34.661883,9,56,medio


In [ ]:
df_har.head()

,sample_id,csi_mean,csi_max,csi_min,csi_std,num_filas,num_columnas,rango_csi
0,1,92.915050,176.139150,17.464249,41.452590,9,56,medio
1,2,87.472571,165.423698,26.683328,33.616029,9,56,medio
2,3,94.036743,172.699160,23.086793,38.146070,9,56,medio
3,4,99.896125,166.207701,36.055513,31.917030,9,56,medio
4,5,110.250978,184.390889,40.049969,34.661883,9,56,medio


In [ ]:
df_har.isnull().any()

sample_id       False
csi_mean        False
csi_max         False
csi_min         False
csi_std         False
num_filas       False
num_columnas    False
rango_csi       False
dtype: bool

In [ ]:
df_har.isnull().sum()

sample_id       0
csi_mean        0
csi_max         0
csi_min         0
csi_std         0
num_filas       0
num_columnas    0
rango_csi       0
dtype: int64

In [ ]:
estadisticas_csi_mean = {
    "promedio_csi_mean": df_har["csi_mean"].mean(),
    "maximo_csi_mean": df_har["csi_mean"].max(),
    "minimo_csi_mean": df_har["csi_mean"].min()
}

estadisticas_csi_mean

{'promedio_csi_mean': np.float64(96.66003269716934),
 'maximo_csi_mean': np.float64(164.11420054266793),
 'minimo_csi_mean': np.float64(34.206799813401716)}

In [ ]:
agrupacion_har = df_har.groupby(["rango_csi", "num_filas"])["csi_mean"].agg(["max", "min"])

agrupacion_har

,,max,min
rango_csi,num_filas,,
bajo,9,77.506153,34.206800
medio,9,120.805916,77.509779
alto,9,164.114201,120.813500


## Análisis del DataFrame 3: `df_annotations`

Este DataFrame contiene las etiquetas de actividad humana asociadas a los archivos PCAP del dataset HAR-ORT. 
Incluye el identificador del registro, la actividad realizada, el usuario participante y el nombre del archivo 
de captura. Para facilitar el análisis numérico, se añadió la variable `filename_length`.

In [ ]:
df_annotations["filename_length"] = df_annotations["filename"].str.len()
df_annotations["is_pcap"] = df_annotations["filename"].str.endswith(".pcap")

df_annotations.head()

,id,activity,user,filename,filename_length,is_pcap
0,1,fall,benja,fall_benja_2025-07-12_00-23-38.pcap,35,True
1,2,fall,benja,fall_benja_2025-07-12_00-23-47.pcap,35,True
2,3,fall,benja,fall_benja_2025-07-12_00-23-56.pcap,35,True
3,4,fall,benja,fall_benja_2025-07-12_00-24-05.pcap,35,True
4,5,fall,benja,fall_benja_2025-07-12_00-24-14.pcap,35,True


In [ ]:
df_annotations.head()

,id,activity,user,filename,filename_length,is_pcap
0,1,fall,benja,fall_benja_2025-07-12_00-23-38.pcap,35,True
1,2,fall,benja,fall_benja_2025-07-12_00-23-47.pcap,35,True
2,3,fall,benja,fall_benja_2025-07-12_00-23-56.pcap,35,True
3,4,fall,benja,fall_benja_2025-07-12_00-24-05.pcap,35,True
4,5,fall,benja,fall_benja_2025-07-12_00-24-14.pcap,35,True


In [ ]:
df_annotations.isnull().any()

id                 False
activity           False
user               False
filename           False
filename_length    False
is_pcap            False
dtype: bool

In [ ]:
df_annotations.isnull().sum()

id                 0
activity           0
user               0
filename           0
filename_length    0
is_pcap            0
dtype: int64

In [ ]:
estadisticas_filename = {
    "promedio_filename_length": df_annotations["filename_length"].mean(),
    "maximo_filename_length": df_annotations["filename_length"].max(),
    "minimo_filename_length": df_annotations["filename_length"].min()
}

estadisticas_filename

{'promedio_filename_length': np.float64(34.9652914210871),
 'maximo_filename_length': np.int64(38),
 'minimo_filename_length': np.int64(33)}

In [ ]:
agrupacion_annotations = df_annotations.groupby(["activity", "user"])["filename_length"].agg(["max", "min"])

agrupacion_annotations

max  min
activity user             
fall     benja     35   35
         bruno     35   35
         flor      34   34
         luigi     35   35
         mathias   37   37
         pablo     35   35
         paula     35   35
         peke      34   34
         vero      34   34
quiet    benja     36   36
         bruno     36   36
         enzo      35   35
         flor      35   35
         luigi     36   36
         mathias   38   38
         nico      35   35
         pablo     36   36
         paula     36   36
         peke      35   35
         vero      35   35
sit      benja     34   34
         bruno     34   34
         enzo      33   33
         flor      33   33
         luigi     34   34
         mathias   36   36
         nico      33   33
         pablo     34   34
         paula     34   34
         peke      33   33
         vero      33   33
stand    benja     36   36
         bruno     36   36
         flor      35   35
         luigi     36   36
         mathias   38   38
         nico      35   35
         pablo     36   36
         paula     36   36
         peke      35   35
         vero      35   35
walk     benja     35   35
         bruno     35   35
         enzo      34   34
         flor      34   34
         luigi     35   35
         mathias   37   37
         nico      34   34
         pablo     35   35
         paula     35   35
         peke      34   34
         vero      34   34

La agrupación por actividad y usuario permite identificar cómo se distribuyen las capturas PCAP según 
el tipo de movimiento registrado y el participante. Esta información es útil para comprender la cobertura 
del dataset antes de aplicar procesos posteriores de análisis o modelado.

In [5]:
from pathlib import Path
import pandas as pd

ruta_annotations = Path("data/har_ort/annotations.csv")

df_annotations = pd.read_csv(ruta_annotations)

print("Filas y columnas:", df_annotations.shape)
df_annotations.head()

Filas y columnas: (1527, 4)


,id,activity,user,filename
0,1,fall,benja,fall_benja_2025-07-12_00-23-38.pcap
1,2,fall,benja,fall_benja_2025-07-12_00-23-47.pcap
2,3,fall,benja,fall_benja_2025-07-12_00-23-56.pcap
3,4,fall,benja,fall_benja_2025-07-12_00-24-05.pcap
4,5,fall,benja,fall_benja_2025-07-12_00-24-14.pcap


In [6]:
conteo_capturas = df_annotations.groupby(["activity", "user"])["id"].count().reset_index()

conteo_capturas = conteo_capturas.rename(columns={
    "id": "cantidad_capturas"
})

conteo_capturas

,activity,user,cantidad_capturas
0,fall,benja,30
1,fall,bruno,30
2,fall,flor,30
3,fall,luigi,25
4,fall,mathias,30
5,fall,pablo,25
6,fall,paula,20
7,fall,peke,30
8,fall,vero,20
9,quiet,benja,30


In [7]:
resumen_capturas_actividad = df_annotations.groupby("activity")["id"].count().reset_index()

resumen_capturas_actividad = resumen_capturas_actividad.rename(columns={
    "id": "total_capturas"
})

resumen_capturas_actividad

,activity,total_capturas
0,fall,240
1,quiet,330
2,sit,330
3,stand,297
4,walk,330


## Aplicación Profesional de la Práctica: 

`Manuel Vicente`


Actualmente me desempeño como analista de seguridad de la información, con enfoque en la prevención de fuga de información. Dentro de esta área, uno de mis intereses principales es especializarme en análisis de comportamiento de usuarios, conocido como UEBA, debido a que el comportamiento del usuario es un factor esencial para identificar actividades anómalas, prevenir incidentes internos y fortalecer los controles de protección de datos.

En este entorno se generan y gestionan distintos tipos de datos provenientes de herramientas de seguridad, tales como reportes de soluciones DLP, alertas del SOC, eventos de antivirus, registros de actividad de usuarios, información sobre dispositivos, aplicaciones utilizadas, canales de transferencia de archivos y eventos relacionados con posibles incumplimientos de políticas internas. Estos datos, analizados de manera aislada, pueden ofrecer información limitada; sin embargo, al integrarlos correctamente, permiten obtener una visión más completa del riesgo asociado a usuarios, equipos y procesos.

PostgreSQL podría utilizarse para almacenar de forma estructurada los eventos provenientes de diferentes fuentes de seguridad. Docker permitiría disponer de un entorno controlado, portable y reproducible para ejecutar la base de datos y otros componentes del proceso. Python, mediante librerías como pandas y SQLAlchemy, facilitaría la extracción, transformación, carga y análisis de la información. Con estas herramientas sería posible correlacionar eventos de DLP, SOC y antivirus, identificando patrones que no serían evidentes al revisar cada fuente por separado.

La implementación de un proceso ETL aportaría beneficios importantes en mi área de trabajo, principalmente en la mejora de procesos, la automatización del análisis, la agilización de revisiones y la validación de falsos positivos. También permitiría depurar información, normalizar campos, consolidar reportes y generar indicadores más precisos para la toma de decisiones.
Mediante el análisis de la información obtenida se podrían resolver problemas relacionados con la afinación de reglas actuales, reducción de ruido operativo, identificación de comportamientos sospechosos y priorización de eventos relevantes. Esto ayudaría a mejorar los controles existentes, minimizar falsos positivos y enfocar los esfuerzos del equipo de seguridad en eventos que realmente representen un riesgo para la organización.


`ELVIS BORBOR`

De acuerdo a lo aprendido en el módulo de Inteligencia de Negocios en esta parte introductoria transferido a nuestra vida profesional nos enfoca hacia una analítica de datos que toda organización debe tener como objetivo. En la organización donde desempeño mis funciones día a día se genera un sin número de información que son generadas por las diferentes áreas, información como transacciones, operaciones de créditos a corto, mediano o largo plazo, órdenes de pago,  procesos de auditoria, procesos de respaldos, actividades de administración, generación de índices de desempeño, numero de tickets de incidencias, requerimientos, ordenes de cambio, y demás datos que deben procesarse por las áreas indicadas para que esta información sea transformada en una analítica de datos que determine directrices y/o métodos eficaces de transformación para que los altos directivos tomen decisiones acertadas que beneficien a la organización.

El área en donde desarrollo mis actividades es el área de Medios Tecnológicos sección Soporte a usuarios en la que a través de tickets de atención se logra atender requerimientos e incidentes de tecnología reportados en el sistema de Mesa de Servicios. Este sistema a su vez está alimentada con datos de los empleados, como nombres, área de trabajo, extensión, localización, equipo de tecnología reportado; bajo esta perspectiva el área de medios tecnológicos posee una estadística del número de incidentes atendidos, requerimientos levantados de Medios Tecnológicos, número de empleados asistidos, que área reporta más incidentes o requerimientos bajo esta analítica los directivos toman decisiones oportunas y llevan al área de Tecnología a transformar toda la data generada en directrices y capacidades que fomenten la producción en toda la organización. 

En nuestro entorno se genera datos como nombres de empleados, su cargo, localización, numero de contacto, equipo afectado de tecnología, área a la que pertenece. Todos estos datos se mencionan por ejemplo como campo obligatorio en la creación de un ticket de atención sea para reportar una incidencia de TI o pedir un requerimiento de Tecnología.

De mi parte me atrevería a indicar que si se podría utilizar PostgreSQL, Docker y Python en nuestra área con el fin de tabular y realizar el análisis efectivo de por ejemplo número de incidentes atendidos, tipos de soluciones entregadas, numero requerimientos de tecnología como instalación, desinstalación de software, el funcionario que más tickets reporta, segmentar por área que parte de nuestra organización se sirve del área de Medios Tecnológicos.

Traería demasiados beneficios si se realizar el proceso de ETL en nuestra organización pues en una de sus formas se tendría mejor canalizados los recursos tanto de equipos como de personal asignado al área de TI. Permitiría a los directivos de la empresa sacar una estadística precisa del área de Medios Tecnológicos y con ello tomar decisiones oportunas para dar la solución eficaz y eficiente que el usuario final necesita de parte de MT.

Decisiones como el número de empleados que se necesita para el área de Medios Tecnológicos para operar, número de equipos de tecnología que necesita la organización para funcionar. Si es necesario contratar algún outsourcing para ciertos tipos de atenciones de tecnología, también para realizar auditorías a mediano plazo.


`Klever Barahona`

Actualmente me desempeño como Analista Informático en la Dirección Nacional de Seguridad de la Información y Protección de Datos (DNSIPD) del Instituto Ecuatoriano de Seguridad Social (IESS), área encargada de promover el cumplimiento de controles de seguridad de la información, protección de datos personales, gestión de riesgos y seguimiento normativo dentro de la institución.

En este entorno se generan y gestionan diversos tipos de datos relacionados con activos de información, matrices de riesgos, registros de actividades de tratamiento (RAT), incidentes de seguridad, evaluaciones de impacto en protección de datos, controles de cumplimiento, inventarios tecnológicos y reportes generados por diferentes dependencias institucionales. Estos datos provienen de múltiples fuentes, incluyendo bases de datos, hojas de cálculo, sistemas institucionales y documentos electrónicos.

Los conocimientos adquiridos durante esta práctica podrían aplicarse directamente para mejorar los procesos de integración y análisis de información dentro de la organización. PostgreSQL permitiría centralizar y almacenar grandes volúmenes de datos provenientes de distintas áreas institucionales, garantizando consistencia, disponibilidad y facilidad de consulta. Docker facilitaría la implementación de ambientes controlados y reproducibles para bases de datos y aplicaciones analíticas, permitiendo desplegar soluciones de forma rápida y segura. Por su parte, Python proporcionaría herramientas para automatizar tareas de extracción, transformación y análisis de datos, utilizando librerías especializadas para procesamiento de información y generación de reportes.

La implementación de un proceso ETL aportaría importantes beneficios a la institución, ya que permitiría consolidar información dispersa en diferentes formatos y sistemas, mejorar la calidad de los datos mediante procesos de limpieza y validación, reducir tiempos de procesamiento y disponer de información actualizada para la toma de decisiones. Además, facilitaría la construcción de repositorios centralizados que apoyen iniciativas de inteligencia de negocio y análisis institucional.
A través del análisis de la información obtenida sería posible identificar tendencias relacionadas con riesgos de seguridad, niveles de cumplimiento normativo, estado de avance de proyectos institucionales y comportamiento de indicadores estratégicos. Esto permitiría priorizar acciones de mitigación, optimizar recursos, fortalecer controles de seguridad y apoyar la toma de decisiones basada en evidencia. En consecuencia, la aplicación de tecnologías como PostgreSQL, Docker, Python y procesos ETL contribuiría significativamente a mejorar la gestión de la información y la eficiencia operativa dentro del IESS.

